# Competição de Carteira de Investimentos 2026 - PPGOLD

Este notebook tem como objetivo documentar a estratégia, a seleção de ativos e a ponderação da carteira para a competição da disciplina de **Análise de Investimentos**.

## Estratégia Adotada

Com base no conteúdo da disciplina, especialmente no **Módulo 01 (Teoria de Markowitz - Fronteira Eficiente)** e **Módulo 04 (Variáveis de Firma e Valuation)**, a estratégia adotada será:
1. **Seleção de Ativos (Screening fundamentalista):** Escolhemos 4 empresas sólidas na B3, com boa liquidez (regra do edital) e de diferentes setores para garantir descorrelação.
2. **Ponderação (Otimização de Markowitz):** Utilizaremos dados históricos para encontrar o portfólio tangente (Máximo Índice de Sharpe), maximizando o retorno ajustado ao risco frente ao CDI.
3. **Extração de Indicadores:** Coletaremos indicadores de Rentabilidade, Valuation e Endividamento para compor o Relatório Técnico exigido no formulário da competição.

In [ ]:
# Instalação e importação das bibliotecas necessárias
# !pip install yfinance pandas numpy matplotlib seaborn

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## 1. Escolha dos 4 Ativos

Para atender aos critérios de liquidez e representarem setores essenciais da economia brasileira, selecionamos:
* **ITUB4.SA** (Setor Financeiro) - Operações maduras e alta lucratividade (ROE alto).
* **WEGE3.SA** (Bens Industriais) - Tese de crescimento (Growth) com prêmio de Valuation (P/L e P/VP mais altos).
* **VALE3.SA** (Materiais Básicos) - Foco em exportação e pagamento de dividendos (alta exposição ao dólar, ajudando na diversificação).
* **PETR4.SA** (Energia / Óleo e Gás) - Geração de caixa forte (bom endividamento) e dividendos.

In [ ]:
# Definindo os tickers
ativos = ['ITUB4.SA', 'WEGE3.SA', 'VALE3.SA', 'PETR4.SA']

## 2. Extração dos Indicadores Fundamentalistas (Exigência do Edital)

O edital **exige** que o relatório contenha:
1. Indicador de rentabilidade/lucratividade (usaremos o **ROE**)
2. Indicador de valuation (usaremos **P/L** e **P/VP**)
3. Indicador de endividamento (usaremos **Dívida/Patrimônio** ou **Debt to Equity**)

In [ ]:
indicadores = []

for ativo in ativos:
    ticker = yf.Ticker(ativo)
    info = ticker.info
    
    indicadores.append({
        'Ativo': ativo,
        'ROE (Rentabilidade)': info.get('returnOnEquity', 'N/A'),
        'ROA (Rentabilidade)': info.get('returnOnAssets', 'N/A'),
        'P/L (Valuation)': info.get('trailingPE', 'N/A'),
        'P/VP (Valuation)': info.get('priceToBook', 'N/A'),
        'Debt/Equity (Endividamento)': info.get('debtToEquity', 'N/A')
    })

df_indicadores = pd.DataFrame(indicadores)
display(df_indicadores)

## 3. Coleta de Preços Históricos e Cálculo dos Retornos e Covariância

In [ ]:
# Preços históricos dos últimos 3 a 5 anos
data_init = '2021-01-01'
data_fim = '2025-01-01'  # Dados de um período completo anterior para calibrar o modelo

df_precos = yf.download(ativos, start=data_init, end=data_fim)['Adj Close']

# Retorno logarítmico diário
retornos = np.log(df_precos / df_precos.shift(1)).dropna()

# Retornos e Matriz de covariância anualizados
retornos_medios_anuais = retornos.mean() * 252
cov_matrix_anual = retornos.cov() * 252

print("Retornos Esperados Anuializados:")
display(retornos_medios_anuais)

print("\nMatriz de Covariância Anualizada:")
display(cov_matrix_anual)

## 4. Otimização de Carteira de Markowitz (Fronteira Eficiente)

O objetivo da competição é superar o CDI. Vamos assumir um CDI em torne de 10.75% a.a (variável `rf` - risk-free rate). 
Utilizaremos simulações de Monte Carlo para encontrar a carteira que maximiza a razão de Sharpe.

In [ ]:
num_portfolios = 30000
rf = 0.1075 # Taxa CDI estimada como risk-free

resultados = np.zeros((3, num_portfolios))
pesos_registrados = []

for i in range(num_portfolios):
    pesos = np.random.random(len(ativos))
    pesos /= np.sum(pesos)
    pesos_registrados.append(pesos)
    
    # Retorno esperado do portfólio
    retorno_portfolio = np.sum(retornos_medios_anuais * pesos)
    
    # Risco (Desvio Padrão) esperado do portfólio
    risco_portfolio = np.sqrt(np.dot(pesos.T, np.dot(cov_matrix_anual, pesos)))
    
    # Índice de Sharpe
    sharpe_ratio = (retorno_portfolio - rf) / risco_portfolio
    
    resultados[0,i] = risco_portfolio # Volatilidade (Eixo X)
    resultados[1,i] = retorno_portfolio # Retorno (Eixo Y)
    resultados[2,i] = sharpe_ratio

# Encontra o índice da carteira com maior Índice de Sharpe
max_sharpe_idx = np.argmax(resultados[2])
pesos_otimos = pesos_registrados[max_sharpe_idx]

print("---- PESOS ÓTIMOS (MÁXIMO SHARPE) ----")
for ativo, peso in zip(ativos, pesos_otimos):
    print(f"{ativo}: {peso*100:.2f}%")

print("\nRetorno Esperado do Portfólio:", round(resultados[1,max_sharpe_idx]*100, 2), "%")
print("Volatilidade do Portfólio:", round(resultados[0,max_sharpe_idx]*100, 2), "%")
print("Índice de Sharpe Esp.:", round(resultados[2,max_sharpe_idx], 4))

## 5. Visualização da Fronteira Eficiente

In [ ]:
plt.figure(figsize=(10, 6))
# Gráfico de dispersão de todos os portfólios simulados
plt.scatter(resultados[0,:], resultados[1,:], c=resultados[2,:], cmap='viridis', marker='o', s=10, alpha=0.3)
plt.colorbar(label='Sharpe Ratio')

# Marcando a carteira com o maior Índice de Sharpe
plt.scatter(resultados[0, max_sharpe_idx], resultados[1, max_sharpe_idx], color='red', marker='*', s=200, label='Máximo Índice de Sharpe')

plt.title('Fronteira Eficiente Simulação de Portfólios')
plt.xlabel('Risco Anualizado (Volatilidade)')
plt.ylabel('Retorno Anualizado')
plt.legend(labelspacing=0.8)
plt.show()

## 6. Conclusões para o Relatório Técnico

**1. Justificativa da Alocação:** Com os pesos definidos acima, utilizamos a Teoria Moderna de Portfólios (Markowitz), balanceando matrizes de covariância históricas com os retornos esperados, buscando extrair a carteira com o **Máximo Prêmio de Risco** ante o CDI (maior Índice de Sharpe).

**2. Indicadores:** Conforme os outputs capturados, temos evidências claras dos indicadores da carteira:
- **Rentabilidade:** Avaliamos pelo ROE os lucros das companhias frente ao Patrimônio investido pelos sócios (equity).
- **Valuation:** Métricas de P/L sinalizam se o ativo está mais "descontado" ou refletindo expectativa de forte crescimento (como muitas vezes notado na WEGE3).
- **Endividamento:** O índice global de Dívida/Equity comprova a saúde financeira frente a taxas de juros (CDI altos).

Com esses pesos (normalizados para totalizar exatamente 100%), este é um excelente portfólio final para competir. Basta aplicar esses percentuais nos ativos via o Formulário do MS Teams do Professor Claudio Marcelo Edwards Barros.